# Zero-shot learning (ZSL) – метод, при котором модель делает предсказания без предварительного обучения на целевых данных

- Модель T0_3B уже обучена на многих задачах (обобщенное знание).
- Мы просто даем ей описание клиента и задаем вопрос, не обучая её на наших данных.
- Модель пытается догадаться, но так как она не знакома с этим датасетом, качество предсказаний низкое.

# 1. Установка зависимостей и датасета

In [ ]:
!pip install transformers torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.4 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
import pandas as pd

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
import torch

In [6]:
df = pd.read_csv('bank-full.csv', sep=';')
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


- age: Возраст. Это числовое значение, представляющее возраст клиента.
- job: Работа. Категориальная переменная, обозначающая сферу занятости клиента.
- marital: Семейное положение. Категориальная переменная, определяющая семейный статус клиента.
- education: Уровень образования. Категориальная переменная, указывающая на уровень образования клиента.
- default: Дефолт по кредиту. Бинарная переменная, показывающая, есть ли у клиента просрочки по кредитам.
- balance: Средний годовой баланс. Числовая переменная, представляющая средний годовой баланс клиента в евро.
- housing: Жилищный кредит. Бинарная переменная, указывающая, есть ли у клиента ипотечный кредит.
- loan: Личный кредит. Бинарная переменная, указывающая, есть ли у клиента личный кредит.
- contact: Тип контакта. Категориальная переменная, указывающая тип контакта с клиентом.
- day_of_week: День недели. Дата переменная, указывающая день недели последнего контакта.

In [8]:
model_name = "bigscience/T0_3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/11.4G [00:00<?, ?B/s]

# 2. Подготовка данных

In [9]:
def concatenate_text_bank(x):
    # Преобразуем бинарные значения в текст
    if x['default'] == 'yes':
      dft = 'has a credit in default'
    else:
        dft = 'has no credit in default'

    if x['housing'] == 'yes':
        hsg = 'has housing loan'
    else:
        hsg = "doesn't have housing loan"

    if x['loan'] == 'yes':
        ln = 'has a personal loan'
    else:
        ln = 'has no personal loan'


    # Формируем текст описания
    full_text = (
        f"This customer is {x['age']} years old, ",
        f"with {x['job']} occupation, ",
        f"is {x['marital']}, ",
        f"with {x['education']} level of education, ",
        f"{dft}, ",
        f"with average yearly balance of {x['balance']} euros, ",
        f"{hsg}, ",
        f"{ln}, ",
        f"contacted via {x['contact']} type of contact, ",
        f"on {x['day']} day of {x['month']}, ",
        f"with the last call duration of {x['duration']} seconds, ",
        f"{x['campaign']} times of call in current marketing campaign, ",
        f"with {x['pdays']} days of pass between contacts, ",
        f" {x['previous']} times of contacts in the previous campaign, ",
        f" and last poutcome is {x['poutcome']}. "
    )
    return ''.join(full_text)

In [10]:
df['text'] = df.apply(lambda x: concatenate_text_bank(x), axis=1)

In [20]:
df['text'].iloc[24]

'This customer is 40 years old, with retired occupation, is married, with primary level of education, has no credit in default, with average yearly balance of 0 euros, has housing loan, has a personal loan, contacted via unknown type of contact, on 5 day of may, with the last call duration of 181 seconds, 1 times of call in current marketing campaign, with -1 days of pass between contacts,  0 times of contacts in the previous campaign,  and last poutcome is unknown. '

In [21]:
df['y'].iloc[24]

'no'

In [22]:
df['y'].value_counts()

,count
y,
no,39922
yes,5289


In [25]:
df_1 = df[['text', 'y']]
df_1.head(3)

,text,y
0,"This customer is 58 years old, with management...",no
1,"This customer is 44 years old, with technician...",no
2,"This customer is 33 years old, with entreprene...",no


In [23]:
def get_zero_shot_predictions(df):
    # Подготовка промпта
    prompt_template = "Does this client subscribe to a term deposit? Yes or no?"

    predictions = []

    for text in df['text']:
        input_text = f"{text} {prompt_template}"

        # Токенизация входных данных
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512).to("cuda")

        # Получение прогноза от модели (в zero-shot режиме)
        outputs = model.generate(**inputs)

        # Декодирование ответа из токенов в текст
        answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Преобразование ответа в бинарное значение (Yes/No -> 1/0)
        if answer_text == 'Yes':
            prediction_label = 1  # 'yes'
            answer_choices_label = 'Yes'
        else:
            prediction_label = 0  # 'no'
            answer_choices_label = 'No'

        predictions.append(prediction_label)

    return predictions

In [28]:
predictions_zero_shot = get_zero_shot_predictions(df)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


In [32]:
predictions_zero_shot[34059]

0

In [33]:
# Подсчет количества каждого класса
class_0_count = predictions_zero_shot.count(0)
class_1_count = predictions_zero_shot.count(1)

print(f"Количество примеров класса 0 (No): {class_0_count}")
print(f"Количество примеров класса 1 (Yes): {class_1_count}")

Количество примеров класса 0 (No): 45211
Количество примеров класса 1 (Yes): 0


In [34]:
# Преобразование реальных меток в бинарный формат
def convert_labels_to_binary(labels):
    return [1 if label == 'yes' else 0 for label in labels]

real_labels_binary = convert_labels_to_binary(df['y'])

In [37]:
accuracy_zs = accuracy_score(real_labels_binary, predictions_zero_shot)
f1_zs = f1_score(real_labels_binary, predictions_zero_shot)
precision_zs = precision_score(real_labels_binary, predictions_zero_shot)
recall_zs = recall_score(real_labels_binary, predictions_zero_shot)

print(f"Accuracy Zero-Shot: {accuracy_zs}")
print(f"F1 Score Zero-Shot: {f1_zs}")
print(f"Precision Zero-Shot: {precision_zs}")
print(f"Recall Zero-Shot: {recall_zs}")

# Матрица ошибок для более детальной информации
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(real_labels_binary, predictions_zero_shot)
print("Матрица ошибок:")
print(conf_mat)

Accuracy Zero-Shot: 0.8830151954170445
F1 Score Zero-Shot: 0.0
Precision Zero-Shot: 0.0
Recall Zero-Shot: 0.0
Матрица ошибок:
[[39922     0]
 [ 5289     0]]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [38]:
def zero_shot_predict(text):
    prompt = f"Classify if this bank customer will subscribe a term deposit.\nText: {text}\nOptions: yes/no."

    # Токенизируем и отправляем на GPU
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(input_ids)

    predicted_class_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

    return 'yes' if predicted_class_text.lower() == 'yes' else 'no'


In [ ]:
# Получение предсказаний на тестовом наборе
predict_zero_shot = [zero_shot_predict(text) for text in df['text']]

In [ ]:
# Вычисление метрик качества
y_true_zero_shot_labels_ints = [1 if y == "yes" else 0 for y in y_test]
y_pred_zero_shot_labels_ints = [1 if y == "yes" else 0 for y in predictions_zero_shot]

roc_auc_zero_shot = roc_auc_score(y_true_zero_shot_labels_ints, y_pred_zero_shot_labels_ints)
f1_zero_shot = f1_score(y_true_zero_shot_labels_ints, y_pred_zero_shot_labels_ints)
accuracy_zero_shot = accuracy_score(y_true_zero_shot_labels_ints, y_pred_zero_shot_labels_ints)
precision_zero_shot = precision_score(y_true_zero_shot_labels_ints, y_pred_zero_shot_labels_ints)
recall_zero_shot = recall_score(y_true_zero_shot_labels_ints, y_pred_zero_shot_labels_ints)

print(f"Zero-Shot ROC AUC: {roc_auc_zero_shot}")
print(f"Zero-Shot F1 Score: {f1_zero_shot}")
print(f"Zero-Shot Accuracy: {accuracy_zero_shot}")
print(f"Zero-Shot Precision: {precision_zero_shot}")
print(f"Zero-Shot Recall: {recall_zero_shot}")

Zero-Shot ROC AUC: 0.5198487712665406
Zero-Shot F1 Score: 0.07636363636363637
Zero-Shot Accuracy: 0.8876479044564857
Zero-Shot Precision: 1.0
Zero-Shot Recall: 0.03969754253308128


Выводы:
- Модель почти всегда предсказывает "no", потому что большинство клиентов в датасете действительно не подписываются (классы несбалансированы).
- ROC AUC ≈ 0.52 говорит о том, что модель почти случайно различает классы.
- F1 = 0.0764 означает, что модель совершенно не умеет находить подписавшихся клиентов.
- Текущие результаты слишком низкие, значит, нужно fine-tuning (дообучение модели) на этих данных.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['y'],
                                                    test_size=0.2, random_state=42, stratify=df['y'])